# 📊 Benchmark End-to-End: Toàn bộ Pipeline Traffic RAG
**Mục tiêu:** Chạy toàn bộ luồng `AgenticTrafficChat` (Classify → Rewrite → Retrieve → Generate) trên bộ câu hỏi Ground Truth và đo lường tự động các chỉ số:

| Chỉ số | Ý nghĩa |
|--------|---------|
| **Routing Accuracy** | Router có phân loại đúng loại câu hỏi không? |
| **Retrieval Precision@5** | Trong top-5 kết quả có chứa điều luật đúng không? |
| **Generation Faithfulness** | Câu trả lời có trích dẫn đúng số Điều/NĐ không? |
| **Latency** | Thời gian phản hồi trung bình |

In [1]:
import os, sys, csv, json, time, yaml
from dotenv import load_dotenv

# 1. Thiết lập đường dẫn
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
BASE_RESEARCH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
load_dotenv(os.path.join(PROJECT_ROOT, "..", ".env"))

# 2. Import core modules
from source.core.config import Settings
from source.generation.rag_pipeline import TrafficRAGPipeline
from source.evaluation.evaluator_ragas import RagasEvaluator
from source.retrieval.hybrid_retriever import HybridRetriever
from rank_bm25 import BM25Okapi
import google.generativeai as genai

# 3. Khởi tạo
settings = Settings()
api_key = settings.api_key or os.getenv("API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-flash-latest")

# Khởi tạo Retriever
retriever = HybridRetriever(settings=settings, collection_name="Traffic_Law_Hybrid")

# Load chunks
CHUNKS_PATH = os.path.abspath(os.path.join(PROJECT_ROOT, "Data", "chunks", "traffic_chunks.json"))
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    retriever.corpus_chunks = json.load(f)
retriever.bm25 = BM25Okapi([retriever._tokenize(c["content"]) for c in retriever.corpus_chunks])

# Load prompts
PROMPT_PATH = os.path.abspath(os.path.join(PROJECT_ROOT, "source", "core", "traffic_prompts.yaml"))
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompts = yaml.safe_load(f)["prompts"]

# Pipeline để dùng ở các cell Ragas
pipeline = TrafficRAGPipeline(settings)

print(f" Môi trường sẵn sàng! {len(retriever.corpus_chunks):,} chunks luật đã được nạp.")

/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/generation/rag_pipeline.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as poss

🔄 Đang sử dụng API Key thứ 1: AIzaSyA1...
 Đang tải mô hình nhúng: KeepItReal/vietnamese-sbert...


/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/retrieval/hybrid_retriever.py:14: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  self.client = QdrantClient(host=self.settings.qdrant_host, port=self.settings.qdrant_port)
/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/model/embedding_model.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings_bkai = HuggingFaceEmbeddings(


Đã khởi tạo Embedding Model thành công!
✅ Hệ thống sẵn sàng với 5 API Keys!


## 1. Load bộ Ground Truth

In [1]:
GT_PATH = os.path.join(BASE_RESEARCH, 'Eval_System', 'dataset', 'ground_truth_traffic.csv')

ground_truth = []
with open(GT_PATH, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ground_truth.append(row)

print(f" Đã tải {len(ground_truth)} câu hỏi ground truth")
print(f"\n Phân phối danh mục:")
from collections import Counter
cats = Counter(r['category'] for r in ground_truth)
for cat, cnt in cats.most_common():
    print(f"   {cat}: {cnt}")


NameError: name 'os' is not defined

## 2. Định nghĩa các hàm gọi pipeline

In [ ]:
# Các hàm này đã được tích hợp vào class TrafficRAGPipeline để hỗ trợ xoay vòng Key tự động.
print("Sử dụng pipeline.run(query) thay thế.")

✅ Các hàm pipeline E2E đã sẵn sàng (có retry tự động khi 429)!


## 3. Chạy Benchmark

In [ ]:
import os, json, time

CHECKPOINT_FILE = "dataset/benchmark_results.json"

eval_results = []

if os.path.exists(CHECKPOINT_FILE):

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:

        eval_results = json.load(f)

    print(f"📂 Tìm thấy {len(eval_results)} kết quả cũ. Đang chạy tiếp...")



processed_questions = {res["question"] for res in eval_results}

print(f"🚀 Bắt đầu Benchmark trên {len(ground_truth)} câu hỏi...")



for i, gt_row in enumerate(ground_truth):

    query = gt_row["question"]

    if query in processed_questions: continue

    

    print(f"[{i+1:02d}/{len(ground_truth)}] Đang xử lý: '{query[:50]}'...")

    try:

        result = pipeline.run(query)

        # Thu thập đầy đủ thông tin cho Ragas

        eval_results.append({

            "question": query,

            "answer": result.get("answer", ""),

            "contexts": [r["chunk"]["content"] for r in result.get("references", [])],

            "ground_truth": gt_row.get("expected_answer", "")

        })

        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:

            json.dump(eval_results, f, ensure_ascii=False, indent=2)

        print(f"   ✅ Xong câu {i+1}!")

    except Exception as e:

        print(f"   ❌ Lỗi: {e}")

        break

    time.sleep(2)

print("✅ Bước 1: Thu thập kết quả Benchmark hoàn tất!")


🚀 Bắt đầu Benchmark End-to-End trên 10 câu hỏi...
   (Mỗi câu hỏi cần ~5-10 giây do gọi Gemini nhiều lần)

[01/10] Đang xử lý: 'Người điều khiển xe máy có nồng độ cồn vượt 0,25mg/l kh'...
   ⏳ Rate limit. Chờ 39s rồi thử lại (lần 1/10)...
   ⏳ Rate limit. Chờ 59s rồi thử lại (lần 2/10)...
   ⏳ Rate limit. Chờ 60s rồi thử lại (lần 3/10)...
   ⏳ Rate limit. Chờ 59s rồi thử lại (lần 4/10)...
   ⏳ Rate limit. Chờ 60s rồi thử lại (lần 5/10)...
   ⏳ Rate limit. Chờ 58s rồi thử lại (lần 1/10)...


## 4. Tổng kết kết quả

In [ ]:
from collections import defaultdict

print("\n" + "="*60)
print("📊 KẾT QUẢ BENCHMARK END-TO-END")
print("="*60)

n = len(eval_results)
routing_acc = sum(r['routing_correct'] for r in eval_results) / n
p_at_5_avg = sum(r['precision_at_5'] for r in eval_results) / n
faithfulness = sum(r['faithful'] for r in eval_results) / n
avg_latency = sum(r['latency'] for r in eval_results) / n

print(f"  📌 Số câu hỏi test         : {n}")
print(f"  🚦 Routing Accuracy         : {routing_acc:.1%}")
print(f"  🔍 Retrieval Precision@5    : {p_at_5_avg:.1%}")
print(f"  ✍️  Generation Faithfulness  : {faithfulness:.1%}")
print(f"  ⏱️  Latency Trung bình       : {avg_latency:.1f}s/câu")

# Phân tích theo danh mục
print(f"\n📋 Phân tích theo danh mục:")
cat_stats = defaultdict(lambda: {'total': 0, 'p_hit': 0})
for r in eval_results:
    cat_stats[r['category']]['total'] += 1
    cat_stats[r['category']]['p_hit'] += r['precision_at_5']

for cat, stats in cat_stats.items():
    prec = stats['p_hit'] / stats['total']
    bar = '█' * int(prec * 10)
    print(f"   {cat:20s}: P@5={prec:.1%}  {bar}")



📊 KẾT QUẢ BENCHMARK END-TO-END
  📌 Số câu hỏi test         : 10
  🚦 Routing Accuracy         : 0.0%
  🔍 Retrieval Precision@5    : 0.0%
  ✍️  Generation Faithfulness  : 0.0%
  ⏱️  Latency Trung bình       : 0.0s/câu

📋 Phân tích theo danh mục:
   Nong_Do_Con         : P@5=0.0%  
   Toc_Do              : P@5=0.0%  
   Den_Tin_Hieu        : P@5=0.0%  
   Mu_Bao_Hiem         : P@5=0.0%  
   Bao_Hiem            : P@5=0.0%  
   GPLX                : P@5=0.0%  
   Dung_Do             : P@5=0.0%  
   Den_Chieu_Sang      : P@5=0.0%  


In [ ]:
# Các câu trả lời sai (Precision@5 = 0)
failures = [r for r in eval_results if r['precision_at_5'] == 0]
if failures:
    print(f"\n❌ {len(failures)} câu hỏi RETRIEVAL MISS (không tìm thấy điều luật đúng):")
    for f in failures:
        print(f"  Q: '{f['question']}'")
        print(f"     Kỳ vọng: {f['expected_article']} | Danh mục: {f['category']}")
        print()
else:
    print("\n🎉 Tất cả câu hỏi đều được truy xuất đúng điều luật!")

# Export kết quả
output_path = os.path.join(BASE_RESEARCH, 'Eval_System', 'dataset', 'benchmark_results.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)
print(f"\n💾 Đã lưu kết quả chi tiết vào: {output_path}")



❌ 10 câu hỏi RETRIEVAL MISS (không tìm thấy điều luật đúng):
  Q: 'Người điều khiển xe máy có nồng độ cồn vượt 0,25mg/l khí thở bị phạt bao nhiêu tiền?'
     Kỳ vọng: Điều 6 | Danh mục: Nong_Do_Con

  Q: 'Xe máy vượt tốc độ quy định từ 10km/h đến dưới 20km/h bị phạt bao nhiêu?'
     Kỳ vọng: Điều 6 | Danh mục: Toc_Do

  Q: 'Ô tô vượt đèn đỏ tại ngã tư bị phạt thế nào?'
     Kỳ vọng: Điều 5 | Danh mục: Den_Tin_Hieu

  Q: 'Không đội mũ bảo hiểm khi đi xe máy phạt bao nhiêu?'
     Kỳ vọng: Điều 11 | Danh mục: Mu_Bao_Hiem

  Q: 'Xe ô tô không có bảo hiểm trách nhiệm dân sự bắt buộc bị phạt gì?'
     Kỳ vọng: Điều 21 | Danh mục: Bao_Hiem

  Q: 'Người điều khiển xe máy không có giấy phép lái xe bị phạt bao nhiêu?'
     Kỳ vọng: Điều 21 | Danh mục: GPLX

  Q: 'Xe máy dừng đỗ trên cầu có bị phạt không?'
     Kỳ vọng: Điều 9 | Danh mục: Dung_Do

  Q: 'Đi xe máy ban đêm không bật đèn chiếu sáng phạt bao nhiêu?'
     Kỳ vọng: Điều 18 | Danh mục: Den_Chieu_Sang

  Q: 'Xe tải chạy quá tốc độ quy đ

## 5. So sánh có Rewriting vs không Rewriting

In [ ]:
# Chạy lại không có rewriting để so sánh với kết quả trên
print("🔄 Đang chạy lại KHÔNG có Query Rewriting để so sánh...")

no_rewrite_results = []
for gt_row in ground_truth[:5]:  # Chỉ chạy 5 câu đầu để tiết kiệm thời gian
    query = gt_row['question']
    expected = gt_row['expected_article']
    
    result_no_rw = run_pipeline_e2e(query, use_rewrite=False)
    retrieved = result_no_rw.get('retrieved_docs', [])
    p5 = 0.0
    for r in retrieved[:5]:
        if expected.lower() in r['chunk']['metadata'].get('dieu','').lower():
            p5 = 1.0; break
    no_rewrite_results.append({'question': query, 'p5': p5})

# Lấy cùng 5 câu từ kết quả rewriting
rw_p5 = [eval_results[i]['precision_at_5'] for i in range(min(5, len(eval_results)))]
no_rw_p5 = [r['p5'] for r in no_rewrite_results]

print(f"\n{'='*60}")
print(f"{'Câu hỏi':<45} {'Không RW':>10} {'Có RW':>8}")
print(f"{'-'*60}")
for i in range(len(no_rewrite_results)):
    q = ground_truth[i]['question'][:43]
    no_rw = '✅' if no_rw_p5[i] > 0 else '❌'
    rw = '✅' if rw_p5[i] > 0 else '❌'
    print(f"{q:<45} {no_rw:>10} {rw:>8}")

avg_no_rw = sum(no_rw_p5) / len(no_rw_p5)
avg_rw = sum(rw_p5) / len(rw_p5)
print(f"\n  Trung bình P@5 Không Rewrite: {avg_no_rw:.1%}")
print(f"  Trung bình P@5 Có Rewrite   : {avg_rw:.1%}")
print(f"  {'→ Query Rewriting CẢI THIỆN kết quả! 🎉' if avg_rw >= avg_no_rw else '→ Query Rewriting KHÔNG cải thiện, cần xem lại prompt!'}")


🔄 Đang chạy lại KHÔNG có Query Rewriting để so sánh...


NotFound: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

---
## 6. 📊 Nâng cao: Đánh giá bằng RAGAS (LLM-as-a-Judge)

Mục này sử dụng thư viện **Ragas** để đo 4 chỉ số chất lượng RAG:

| Chỉ số | Ý nghĩa |
|--------|------|
| **Faithfulness** | Câu trả lời có dựa trên context không? (tránh hallucination) |
| **Answer Relevance** | Câu trả lời có đúng trọng tâm câu hỏi không? |
| **Context Precision** | Tài liệu truy xuất có liên quan không? |
| **Context Recall** | Có đủ thông tin để trả lời đúng không? |

> **LangSmith Tracing**: Nếu `LANGCHAIN_API_KEY` đã cấu hình trong `.env`, mọi bước đánh giá sẽ tự động được ghi lên [smith.langchain.com](https://smith.langchain.com/).

In [ ]:
# 📊 Bước 2: Chấm điểm Ragas (Dựa trên dữ liệu đã thu thập ở trên)

RAGAS_REPORT = "dataset/ragas_report.json"

evaluator = RagasEvaluator(settings, model_name="gemini-flash-latest")



# Chấm điểm thông qua bộ evaluator (đã hỗ trợ xoay vòng key)

score = evaluator.evaluate_existing_results(eval_results, output_path=RAGAS_REPORT)



if score:

    import pandas as pd

    df_report = pd.read_json(RAGAS_REPORT)

    print("
🎯 KẾT QUẢ ĐÁNH GIÁ CHI TIẾT:")

    display(df_report.round(3))



### 📈 Radar Chart - Trực quan hóa kết quả RAGAS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'source_research', 'Eval_System', 'dataset', 'ragas_report.json')

if os.path.exists(OUTPUT_PATH):
    df = pd.read_json(OUTPUT_PATH)
    metrics = [c for c in df.columns if c not in ['question', 'answer', 'contexts', 'ground_truth']]
    values  = [df[m].mean() for m in metrics]

    N = len(metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    v_plot = values + values[:1]
    a_plot = angles + angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.fill(a_plot, v_plot, color='#4A90D9', alpha=0.3)
    ax.plot(a_plot, v_plot, color='#4A90D9', linewidth=2, marker='o')
    ax.set_yticklabels([])
    ax.set_ylim(0, 1)
    ax.set_xticks(angles)
    ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics], fontsize=12)
    ax.set_title('🎯 RAG Quality - RAGAS Metrics', fontsize=14, fontweight='bold', pad=25)
    plt.tight_layout()
    plt.savefig('ragas_radar.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n📊 Điểm trung bình từng chỉ số:')
    for m, v in zip(metrics, values):
        bar = '█' * int(v * 20)
        print(f'  {m.replace("_", " ").title():30s}: {v:.3f}  {bar}')
else:
    print('⚠️ Chưa có file ragas_report.json. Hãy chạy cell bên trên trước.')
